# Fine-Tuning: Train + Compare

This section uploads the training file, fine-tunes `gpt-4o-mini`, and compares the fine-tuned model against the baseline.

### Where to add
Insert these cells **after the Gap 2 cells** (after the Baseline vs RAG comparison), but **before the existing Step 4 cells** (before Cell 66).

### Two-phase process
1. **Phase 1 (Cells A-C):** Upload file and start the fine-tuning job. This takes ~10-20 minutes to train.
2. **Phase 2 (Cells D-G):** After training completes, run the comparison. You MUST wait for Phase 1 to finish before running Phase 2.

> **Cost:** Approximately $0.25 for training on gpt-4o-mini with 20 examples.

---
## Phase 1: Upload and Start Fine-Tuning
Run these cells once. Then wait for training to complete before running Phase 2.

In [ ]:
# ── Cell A: Upload Training File to OpenAI ──

training_file = client.files.create(
    file=open("quiz_finetune_train.jsonl", "rb"),
    purpose="fine-tune"
)

print("File uploaded successfully")
print("File ID:", training_file.id)
print("Status:", training_file.status)

# SAVE THIS — you need it for the next cell
TRAINING_FILE_ID = training_file.id

In [ ]:
# ── Cell B: Create Fine-Tuning Job ──

fine_tune_job = client.fine_tuning.jobs.create(
    training_file=TRAINING_FILE_ID,
    model="gpt-4o-mini",
    hyperparameters={
        "n_epochs": 3
    },
    suffix="quiz-gen"  # your fine-tuned model will include this in its name
)

print("Fine-tuning job created!")
print("Job ID:", fine_tune_job.id)
print("Status:", fine_tune_job.status)
print()
print("SAVE THIS JOB ID:", fine_tune_job.id)
print()
print("Training typically takes 10-20 minutes.")
print("Run the next cell to check status. Do NOT proceed to Phase 2 until status = 'succeeded'.")

# SAVE THIS
FINE_TUNE_JOB_ID = fine_tune_job.id

In [ ]:
# ── Cell C: Check Fine-Tuning Status (re-run this until status = 'succeeded') ──

# If you lost the job ID, paste it here:
# FINE_TUNE_JOB_ID = "ftjob-xxxxxxxxxxxxxxxx"

job_status = client.fine_tuning.jobs.retrieve(FINE_TUNE_JOB_ID)

print("Job ID:", job_status.id)
print("Status:", job_status.status)
print("Model:", job_status.fine_tuned_model)
print()

if job_status.status == "succeeded":
    FINE_TUNED_MODEL = job_status.fine_tuned_model
    print("TRAINING COMPLETE!")
    print("Your fine-tuned model:", FINE_TUNED_MODEL)
    print()
    print("You can now run Phase 2 cells below.")
elif job_status.status == "failed":
    print("TRAINING FAILED")
    print("Error:", job_status.error)
elif job_status.status == "running":
    print("Still training... Re-run this cell in a few minutes.")
elif job_status.status == "queued":
    print("Job is queued. Training has not started yet. Re-run this cell in a few minutes.")
else:
    print("Current status:", job_status.status, "— Re-run this cell in a few minutes.")

---
## Phase 2: Compare Fine-Tuned Model vs Baseline
Only run these cells AFTER Cell C shows `status = 'succeeded'`.

In [ ]:
# ── Cell D: Define Fine-Tuned Quiz Generator ──

# If you're running this in a new session, paste your model name here:
# FINE_TUNED_MODEL = "ft:gpt-4o-mini-2024-07-18:your-org::xxxxxxxx"

print("Using fine-tuned model:", FINE_TUNED_MODEL)

def generate_quiz_finetuned(notes, temperature=0.3):
    user_prompt = f"""Create a diagnostic quiz based ONLY on the lecture notes below.

Requirements:
- 3 Multiple Choice Questions (Easy, Medium, Hard)
- 2 Short Answer Questions
- 1 Application Question
- Include answers and explanations
- Do NOT use outside knowledge

Lecture Notes:
{notes}
"""

    response = client.chat.completions.create(
        model=FINE_TUNED_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": HARDENED_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

print("generate_quiz_finetuned() defined")

In [ ]:
# ── Cell E: Run Fine-Tuned vs Baseline Comparison ──

print("=" * 60)
print("BASELINE vs FINE-TUNED MODEL COMPARISON")
print("=" * 60)

ft_comparison_examples = [item for item in test if item["case_type"] == "typical"][:5]
print(f"Comparing on {len(ft_comparison_examples)} test examples\n")

ft_comparison_results = []

for item in ft_comparison_examples:
    print(f"Processing: {item['chunk_id']}...")

    # Generate with baseline
    baseline_output = generate_quiz_baseline(item["notes"], temperature=0.3)

    # Generate with fine-tuned model
    finetuned_output = generate_quiz_finetuned(item["notes"], temperature=0.3)

    # Evaluate both
    baseline_eval = evaluate_quiz_output(item["notes"], baseline_output)
    finetuned_eval = evaluate_quiz_output(item["notes"], finetuned_output)

    ft_comparison_results.append({
        "chunk_id": item["chunk_id"],
        "baseline_output": baseline_output,
        "finetuned_output": finetuned_output,
        "baseline_eval": baseline_eval,
        "finetuned_eval": finetuned_eval
    })

print(f"\nComparison complete: {len(ft_comparison_results)} examples evaluated.")

In [ ]:
# ── Cell F: Fine-Tuned vs Baseline Score Comparison Table ──

ft_baseline_scores = []
ft_finetuned_scores = []

for row in ft_comparison_results:
    b_scores = parse_eval_scores(row["baseline_eval"])
    f_scores = parse_eval_scores(row["finetuned_eval"])
    ft_baseline_scores.append(b_scores)
    ft_finetuned_scores.append(f_scores)

    print("\n" + "=" * 55)
    print(f"CHUNK: {row['chunk_id']}")
    print(f"{'Metric':<25} {'Baseline':>10} {'Fine-Tuned':>12}")
    print("-" * 55)
    for metric in metrics:
        b_val = b_scores.get(metric, "N/A")
        f_val = f_scores.get(metric, "N/A")
        print(f"{metric:<25} {str(b_val):>10} {str(f_val):>12}")

# Average scores
print("\n" + "=" * 60)
print("AVERAGE SCORES: BASELINE vs FINE-TUNED")
print("=" * 60)
print(f"{'Metric':<25} {'Baseline':>10} {'Fine-Tuned':>12} {'Diff':>10}")
print("-" * 60)

for metric in metrics:
    b_vals = [s[metric] for s in ft_baseline_scores if s[metric] is not None]
    f_vals = [s[metric] for s in ft_finetuned_scores if s[metric] is not None]

    b_avg = sum(b_vals) / len(b_vals) if b_vals else 0
    f_avg = sum(f_vals) / len(f_vals) if f_vals else 0
    diff = f_avg - b_avg

    sign = "+" if diff > 0 else ""
    print(f"{metric:<25} {b_avg:>10.2f} {f_avg:>12.2f} {sign}{diff:>9.2f}")

print("-" * 60)
b_overall = sum(sum(s[m] for m in metrics if s[m]) for s in ft_baseline_scores) / (len(ft_baseline_scores) * len(metrics))
f_overall = sum(sum(s[m] for m in metrics if s[m]) for s in ft_finetuned_scores) / (len(ft_finetuned_scores) * len(metrics))
diff_overall = f_overall - b_overall
sign = "+" if diff_overall > 0 else ""
print(f"{'OVERALL AVERAGE':<25} {b_overall:>10.2f} {f_overall:>12.2f} {sign}{diff_overall:>9.2f}")

In [ ]:
# ── Cell G: Full Three-Way Comparison Summary ──

print("=" * 70)
print("COMPLETE COMPARISON: BASELINE vs RAG vs FINE-TUNED")
print("=" * 70)

# Get RAG averages (from Gap 2 results if available)
try:
    rag_avgs = {}
    for metric in metrics:
        r_vals = [s[metric] for s in rag_all_scores if s[metric] is not None]
        rag_avgs[metric] = sum(r_vals) / len(r_vals) if r_vals else 0
    has_rag = True
except:
    has_rag = False
    print("(RAG scores not available — run Gap 2 cells first for full three-way comparison)")

# Baseline averages
baseline_avgs = {}
for metric in metrics:
    b_vals = [s[metric] for s in ft_baseline_scores if s[metric] is not None]
    baseline_avgs[metric] = sum(b_vals) / len(b_vals) if b_vals else 0

# Fine-tuned averages
ft_avgs = {}
for metric in metrics:
    f_vals = [s[metric] for s in ft_finetuned_scores if s[metric] is not None]
    ft_avgs[metric] = sum(f_vals) / len(f_vals) if f_vals else 0

if has_rag:
    print(f"\n{'Metric':<25} {'Baseline':>10} {'RAG':>10} {'Fine-Tuned':>12}")
    print("-" * 62)
    for metric in metrics:
        print(f"{metric:<25} {baseline_avgs[metric]:>10.2f} {rag_avgs[metric]:>10.2f} {ft_avgs[metric]:>12.2f}")
else:
    print(f"\n{'Metric':<25} {'Baseline':>10} {'Fine-Tuned':>12}")
    print("-" * 52)
    for metric in metrics:
        print(f"{metric:<25} {baseline_avgs[metric]:>10.2f} {ft_avgs[metric]:>12.2f}")

print(f"""
{'='*70}

KEY FINDINGS:
- Baseline (prompt-only): Uses hardened system prompt with gpt-4o-mini
- RAG: Retrieves relevant chunks from Qdrant before generating
- Fine-tuned: gpt-4o-mini trained on 20 quiz generation examples

The fine-tuned model has learned the specific quiz format and grounding
patterns from the training examples, which may improve consistency in
output structure. RAG provides dynamic grounding by retrieving the most
relevant content at generation time. Both approaches reduce hallucination
compared to the prompt-only baseline, but through different mechanisms.

Fine-tuned model name: {FINE_TUNED_MODEL}
""")